# FairFace Model Evaluation on MSKCC Dataset (v2)
This notebook evaluates the **FairFace-trained EfficientNet-B4** model on the **MSKCC skin tone dataset**.
**Modes:** Toggle `EVAL_MODE` between `'3-way'` and `'6-way'` in Cell 1.
**Filter:** Excludes dermoscopic images.

In [ ]:
# ==================== CELL 1: Configuration ====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
# --- CONFIGURATION ---
# Toggle: '3-way' or '6-way'
EVAL_MODE = '3-way'
NUM_CLASSES = 3 if EVAL_MODE == '3-way' else 6
IMAGE_SIZE = 380
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- MODEL PATH ---
# Set the correct path for your trained model:
if EVAL_MODE == '3-way':
    MODEL_PATH = "outputs/fairface_3way/best_finetuned_model.pth"
else:
    MODEL_PATH = "outputs/FairFace-Model-2.0/best_finetuned_model.pth"
METADATA_PATH = "datasets/mskcc-skin-tone-labeling-dataset_metadata_2025-11-24.csv"
IMAGE_ROOT = "datasets/MSKCC-images"
print(f"Evaluation Mode: {EVAL_MODE}  |  NUM_CLASSES: {NUM_CLASSES}")
print(f"Model path:      {MODEL_PATH}")
print(f"Using device:    {DEVICE}")

In [ ]:
# ==================== CELL 2: Data Preparation ====================
print("Loading metadata...")
df = pd.read_csv(METADATA_PATH)
# 1. Filter for Close-up Only
df_filtered = df[df['image_type'] != 'dermoscopic'].copy()
df_filtered = df_filtered.dropna(subset=['fitzpatrick_skin_type'])
print(f"Filtered images: {len(df_filtered)}")
# 2. Define Class Mappings based on EVAL_MODE
if EVAL_MODE == '3-way':
    # Training label order: 0=Dark, 1=Medium, 2=Light
    def map_gt(skin_type):
        if skin_type in ['V', 'VI']:   return 'Dark'
        if skin_type in ['III', 'IV']:  return 'Medium'
        if skin_type in ['I', 'II']:    return 'Light'
        return None
    CLASSES = ['Dark', 'Medium', 'Light']  # idx matches training labels
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    # Native 3-way model — probs are already (B, 3)
    def aggregate_probs(probs):
        return probs
else:  # 6-way
    def map_gt(skin_type):
        return skin_type
    # Order: VI -> I (matches training: 0=VI ... 5=I)
    CLASSES = ['VI', 'V', 'IV', 'III', 'II', 'I']
    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    def aggregate_probs(probs_6way):
        return probs_6way  # No change needed
# Apply GT Mapping
df_filtered['target_label'] = df_filtered['fitzpatrick_skin_type'].apply(map_gt)
df_filtered = df_filtered.dropna(subset=['target_label'])
# Add Paths
df_filtered['path'] = df_filtered['isic_id'].apply(lambda x: os.path.join(IMAGE_ROOT, f"{x}.jpg"))
df_filtered = df_filtered[df_filtered['path'].apply(os.path.exists)]
# RESET INDEX is crucial for matching line numbers later
df_filtered = df_filtered.reset_index(drop=True)
print(f"Final test set size: {len(df_filtered)}")
print(f"Classes: {CLASSES}")
print(df_filtered['target_label'].value_counts())

In [ ]:
# ==================== CELL 3: Dataset Class ====================
class InferenceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df['path'].values
        self.labels = df['target_label'].values
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except:
            img = Image.new("RGB", (380, 380))
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_loader = DataLoader(InferenceDataset(df_filtered, test_transform),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# ==================== CELL 4: Model Loading & Inference ====================
# Build model with correct number of output classes
model = models.efficientnet_b4(weights=None)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, NUM_CLASSES)
)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("✓ Model loaded")
else:
    print("❌ Model not found")
model = model.to(DEVICE).eval()
y_true_labels = []
y_scores = []  # Probabilities
print("Running inference...")
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        # Get probabilities (Softmax)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        # Transform probs based on mode
        batch_scores = aggregate_probs(probs)
        y_scores.extend(batch_scores)
        y_true_labels.extend(labels)
y_scores = np.array(y_scores)
# Convert true labels to indices
y_true_indices = np.array([CLASS_TO_IDX[l] for l in y_true_labels])
# Get predictions
y_pred_indices = np.argmax(y_scores, axis=1)
y_pred_labels = [CLASSES[i] for i in y_pred_indices]
print("✓ Inference complete")

In [ ]:
# ==================== CELL 5: Predict & Evaluate ====================
# 1. Overall Accuracy
accuracy = (y_pred_indices == y_true_indices).mean()
print(f"\n{'='*60}")
print(f"TEST SET ACCURACY: {accuracy:.2%}")
print(f"{'='*60}")
# 2. Classification Report
print("\nDetailed Classification Report:")
print(classification_report(y_true_indices, y_pred_indices, target_names=CLASSES))
# 3. Confusion Matrix
cm = confusion_matrix(y_true_indices, y_pred_indices)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Confusion Matrix - {EVAL_MODE} Evaluation')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELL 6: Precision-Recall Curves ====================
y_true_bin = label_binarize(y_true_indices, classes=list(range(NUM_CLASSES)))
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(5 * NUM_CLASSES, 4))
if NUM_CLASSES == 1:
    axes = [axes]
for i, (ax, cls_name) in enumerate(zip(axes, CLASSES)):
    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_scores[:, i])
    ax.plot(recall, precision, lw=2)
    ax.set_title(f'{cls_name}  (AP={ap:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)
plt.suptitle(f'Precision-Recall Curves — {EVAL_MODE}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELL 7: Error Analysis with Images ====================
df_filtered['predicted_label'] = y_pred_labels
df_filtered['correct'] = df_filtered['target_label'] == df_filtered['predicted_label']
# Show worst misclassifications (highest confidence in wrong class)
df_errors = df_filtered[~df_filtered['correct']].copy()
if len(df_errors) > 0:
    # Get confidence of predicted (wrong) class
    wrong_confs = []
    for idx, row in df_errors.iterrows():
        pred_idx = CLASS_TO_IDX.get(row['predicted_label'], 0)
        wrong_confs.append(y_scores[idx][pred_idx])
    df_errors['wrong_confidence'] = wrong_confs
    df_errors = df_errors.sort_values('wrong_confidence', ascending=False)
    n_show = min(8, len(df_errors))
    fig, axes = plt.subplots(2, n_show // 2, figsize=(4 * (n_show // 2), 8))
    axes = axes.flatten()
    for i, (_, row) in enumerate(df_errors.head(n_show).iterrows()):
        try:
            img = Image.open(row['path']).convert('RGB')
            axes[i].imshow(img)
        except:
            pass
        axes[i].set_title(f"GT: {row['target_label']}\nPred: {row['predicted_label']}\nConf: {row['wrong_confidence']:.2f}",
                         fontsize=9)
        axes[i].axis('off')
    plt.suptitle(f'Top {n_show} Misclassifications ({EVAL_MODE})', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications found!")

In [ ]:
# ==================== CELL 8: Confusion Analysis per Class ====================
print("Per-class error breakdown:\n")
for i, cls in enumerate(CLASSES):
    mask_true = y_true_indices == i
    n_total = mask_true.sum()
    n_correct = (y_pred_indices[mask_true] == i).sum()
    n_wrong = n_total - n_correct
    acc = n_correct / n_total if n_total > 0 else 0
    # What was it most confused with?
    if n_wrong > 0:
        wrong_preds = y_pred_indices[mask_true & (y_pred_indices != i)]
        from collections import Counter
        confused_with = Counter(wrong_preds).most_common(3)
        confused_str = ', '.join([f"{CLASSES[c]}: {n}" for c, n in confused_with])
    else:
        confused_str = 'N/A'
    print(f"  {cls:>10s}  |  Support: {n_total:4d}  |  Accuracy: {acc:.1%}  |  Confused with: {confused_str}")

In [ ]:
# ==================== CELL 9: Class-wise Accuracy by Fitzpatrick Type ====================
print("Accuracy broken down by original Fitzpatrick skin type:\n")
fitz_types = sorted(df_filtered['fitzpatrick_skin_type'].unique())
results = []
for ft in fitz_types:
    mask = df_filtered['fitzpatrick_skin_type'] == ft
    subset_true = y_true_indices[mask]
    subset_pred = y_pred_indices[mask]
    n = mask.sum()
    acc = (subset_true == subset_pred).mean() if n > 0 else 0
    results.append({'Fitzpatrick': ft, 'Support': n, 'Accuracy': f'{acc:.1%}'})
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
accs = [float(r['Accuracy'].strip('%')) / 100 for r in results]
bars = ax.bar([r['Fitzpatrick'] for r in results], accs, color=plt.cm.RdYlGn(accs))
ax.set_ylabel('Accuracy')
ax.set_xlabel('Fitzpatrick Type')
ax.set_title(f'Accuracy by Fitzpatrick Type ({EVAL_MODE})')
ax.set_ylim(0, 1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()